In [11]:
import json
import os 

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "romain2021non")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "Romain_et_al_2021.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [12]:
import pandas as pd
import numpy as np
import pyreadstat

df = pd.read_csv(complete_path_1, sep = ';')

df['study_id']="romain2021non"
df.columns = map(str.lower, df.columns)
df=df.applymap(lambda s: s.lower() if type(s) == str else s)


In [13]:
df.rename(columns={"individual": "participant", 
                "species":"species_original",
                "sessions.1":"sessions_1"}, inplace=True)

import re
replace_1=re.compile('(|\(|\)|\:)') 
df.columns = df.columns.str.replace(replace_1, '')
replace_2=re.compile('( |\,)') 
df.columns = df.columns.str.replace(replace_2, '_')
df.columns = df.columns.str.replace('__', '_')


In [14]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
df['participant'] = df['participant'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    df['participant'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)    
df= df.merge(apedf,left_on='participant', right_on='name', how='left')
# df.columns



In [15]:
comp_path_ape_info = os.path.join(pathway_gen, "romain2021non_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)  
apedf = apedf.values.tolist()

for x,y,k in apedf:
    df.loc[df.participant == x, ['species', 'sex']] = y,k 

In [16]:
df.rename(columns={'exchange_1-yes_0-no':'exchange', 
                   'reward_obtained_big_small':'reward_obtained',
                   'visible_proba_of_gain':'visible_probability_of_gain', 
                   'real_proba_of_gain':'real_probability_of_gain',
                   'cover_position_nc_no_cover_cm_cover_in_the_middle_cd_cover_on_the_right_cg_cover_on_the_left':'cover_position',
                    'nbr_of_lr_visible_nbr_of_large_reward_visible':'number_large_visible_rewards',
                    'level_info_predictably_positive_predictably_negative_risky_ambiugous':'level_of_information',
                    'previous_reward_big_small_neutral_intermediate_size':'previous_rewards'}, inplace=True)
exchange_list = [['exchange',1,'yes'],
               ['exchange', 0,'no'],
               ['reward_obtained','s','small'],
               ['reward_obtained','m','medium'],
               ['reward_obtained','b','large'],
               ['cover_position', 'nc','no_cover'],
               ['cover_position', 'cm','cover_middle'],
               ['cover_position', 'cg','cover_left'],
               ['cover_position', 'cd','cover_right'],
               ['level_of_information', 'rsk', 'risk'],
               ['level_of_information', 'pre +', 'predictability_advantageous'],
               ['level_of_information', 'pre -', 'predictability_disadvantageous'],
               ['level_of_information', 'amb', 'ambiguous'],
               ['previous_rewards','s','small'],
               ['previous_rewards','m','medium'],
               ['previous_rewards','b','large'],
               ['previous_rewards','n','none']]
for x,y,z in exchange_list:
    df[x].replace(y, z, inplace=True)

In [17]:
complete_path_age = os.path.join(original_data_pathway, "subject_list.csv")
subject_list = pd.read_csv(complete_path_age)   
df= df.merge(subject_list,left_on='participant', right_on='name', how='left')
df.rename(columns={"age": "age_in_years"}, inplace=True)

In [18]:
studyID_standardized=df[['study_id', 'participant','age_in_years', 'sex', 'species', 'sessions',
                     #      'sessions_1',
                          'trial',
       'lottery', 'exchange', 'reward_obtained',
       'visible_probability_of_gain', 'real_probability_of_gain',
       'cover_position',
       'number_large_visible_rewards',
       'level_of_information',
       'previous_rewards']]
comp_out_path_stand = os.path.join(out_pathway, 'romain2021non_standardized.csv')
studyID_standardized.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)


names =studyID_standardized.columns.tolist()
df = pd.DataFrame(names)
df = df.rename(columns={0: "column_name"})
df["description"] = ""
studyID_glossary=df[["column_name", "description"]]

comp_out_path_glossary = os.path.join(out_pathway, 'romain2021non_glossary.csv')
studyID_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)